# 🗺️ 09 · Dimensionality Reduction and Reconstruction
### From many features to a map you can actually look at: PCA, MDS, Isomap, t-SNE and autoencoders

**Part E · Latent structure** · Module 09 of 29 (core) · builds on 07, 08

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/09_dimensionality_reduction_and_reconstruction.ipynb)

| | |
|---|---|
| **Hardware target** | CPU |
| **Core frameworks** | NumPy, scikit-learn, PyTorch (for the autoencoder) |
| **Prerequisites** | Modules 07 (classical features) and 08 (learned features) |
| **Estimated time** | 90–110 minutes |
| **02-740 topics covered** | feature extraction vs. selection, principal component analysis and eigenimages, classical multidimensional scaling, Isomap, t-SNE, autoencoders as non-linear dimensionality reduction |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every point in every scatter plot below comes from a shape or image that `course_toolkit` generated and whose true class is already known, which is what lets us check each method against ground truth.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the curse of dimensionality and why bioimage analysis specifically benefits from reducing many
  correlated features down to a few meaningful ones.
- Derive PCA as the search for the direction of maximum variance, compute it by eigendecomposition of the
  covariance matrix, and interpret "eigenimages."
- Explain classical multidimensional scaling (MDS) as reconstructing coordinates from a distance matrix,
  and build a simple shape space from pairwise shape distances.
- Explain how Isomap extends MDS with geodesic (manifold) distances, and when that matters.
- Explain the probabilistic idea behind t-SNE and when it reveals structure that PCA misses.
- Train a small autoencoder and explain why it is a non-linear generalization of PCA.

## 🗺️ Contents

1. Why reduce dimensionality in bioimage analysis?
2. Principal component analysis: the maximum-variance direction
3. Eigenimages and PCA for recognition
4. Classical multidimensional scaling (MDS): building a shape space
5. Isomap: following the manifold
6. t-SNE: preserving local neighborhoods
7. Autoencoders: non-linear dimensionality reduction

## 🧫 The biological question

Module 07 already gave us eight numbers per cell (GLCM texture + shape + intensity). A richer pipeline —
CellProfiler-style, say — might compute hundreds. A biologist cannot look at a 200-dimensional scatter
plot, and a classifier trained on hundreds of correlated, partly-redundant features tends to overfit. We
need a map: a small number of new coordinates, ideally two or three, that still separates the biologically
meaningful groups (healthy vs. diseased, treated vs. untreated, shape class vs. shape class) well enough to
look at, cluster, or feed to a simpler downstream classifier.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
torch.manual_seed(90)
SEED = 90
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def glcm_features(image: np.ndarray, levels: int = 8, dx: int = 1, dy: int = 0):
    """Haralick-style texture features from a grey-level co-occurrence matrix (GLCM).

    Quantises ``image`` to ``levels`` grey levels, builds the co-occurrence matrix for the
    offset (dy, dx), and returns a dict of contrast, energy (angular second moment),
    homogeneity and correlation.
    """
    img = image - image.min()
    img = img / (img.max() + 1e-9)
    q = np.clip((img * (levels - 1)).round().astype(int), 0, levels - 1)
    h, w = q.shape
    glcm = np.zeros((levels, levels))
    y0, y1 = max(0, -dy), h - max(0, dy)
    x0, x1 = max(0, -dx), w - max(0, dx)
    a = q[y0:y1, x0:x1]
    b = q[y0 + dy:y1 + dy, x0 + dx:x1 + dx]
    for i, j in zip(a.ravel(), b.ravel()):
        glcm[i, j] += 1
    glcm /= glcm.sum() + 1e-9
    i_idx, j_idx = np.meshgrid(np.arange(levels), np.arange(levels), indexing="ij")
    mu_i, mu_j = (glcm * i_idx).sum(), (glcm * j_idx).sum()
    sig_i = np.sqrt((glcm * (i_idx - mu_i) ** 2).sum()) + 1e-9
    sig_j = np.sqrt((glcm * (j_idx - mu_j) ** 2).sum()) + 1e-9
    contrast = (glcm * (i_idx - j_idx) ** 2).sum()
    energy = (glcm ** 2).sum()
    homogeneity = (glcm / (1 + np.abs(i_idx - j_idx))).sum()
    correlation = ((glcm * (i_idx - mu_i) * (j_idx - mu_j)).sum()) / (sig_i * sig_j)
    return dict(contrast=contrast, energy=energy, homogeneity=homogeneity, correlation=correlation)


def pca(X: np.ndarray, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data. Returns the projected
    scores (N x n_components), the components (n_components x J) and the explained-variance
    ratio of each component."""
    mean = X.mean(axis=0)
    Xc = X - mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (U * S)[:, :n_components]
    explained = (S ** 2) / np.sum(S ** 2)
    return scores, Vt[:n_components], explained[:n_components]


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. Why reduce dimensionality in bioimage analysis?

Dimensionality reduction maps high-dimensional data (many features, or raw pixels) to a low-dimensional
representation while preserving as much meaningful structure as possible. Lower-dimensional representations
help for three concrete reasons: models with fewer parameters are cheaper to fit and less prone to
overfitting (the **curse of dimensionality** — as the number of features grows, the data needed to densely
sample that space grows exponentially, so with a fixed number of cells, a high-dimensional feature space
gets sparser and easier to "overfit" by chance); lower dimensions are far easier to visualize and explore;
and bioimage shape descriptors specifically benefit because a compact "shape space" is often what a
biologist actually wants to reason about.

Two different strategies both reduce dimensionality:

- **Feature selection** keeps a *subset* of the original features, dropping redundant or uninformative
  ones, with no transformation of the kept features (Module 07, Section 5).
- **Feature extraction** builds *new* features as combinations of the originals — a linear combination for
  PCA, a non-linear one for Isomap, t-SNE, or an autoencoder.

Linear methods (PCA, classical MDS) are best when the data lies close to a linear subspace and you care
about preserving *global* structure; non-linear methods (Isomap, t-SNE, autoencoders) try to preserve
*local* neighborhood structure on a possibly curved manifold, at the cost of a less interpretable mapping.

## 2. Principal component analysis: the maximum-variance direction

Given a data matrix $X$ ($N$ samples × $D$ features, mean-centered), PCA looks for the unit direction $w$
that maximizes the variance of the data projected onto it: $\max_w \; w^\top \mathrm{Cov}(X)\, w$ subject
to $\lVert w\rVert=1$. The solution is the eigenvector of the covariance matrix $\mathrm{Cov}(X)$ with the
**largest** eigenvalue; the second principal component is the eigenvector with the next-largest eigenvalue,
constrained to be orthogonal to the first (and hence automatically uncorrelated with it); and so on. The
eigenvalues themselves are the variance *explained* by each component, so $\lambda_k / \sum_j \lambda_j$
is the fraction of total variance the $k$-th component captures.

Equivalently (and numerically more stable), PCA can be computed directly from the singular value
decomposition $X = U S V^\top$: the columns of $V$ are the principal directions, and $US$ are the
projected scores — this is what `course_toolkit.pca` does.

In [ ]:
# ── 2.1 A worked 2-D example: find the first principal component by hand ──
small = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0], [2.3, 2.7], [2.0, 1.6]])
mean = small.mean(axis=0)
centered = small - mean
cov = np.cov(centered, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
print("covariance matrix:\n", np.round(cov, 3))
print("eigenvalues (sorted):", np.round(eigvals, 3))
print("first principal direction (unit vector):", np.round(eigvecs[:, 0], 3))
print("fraction of variance explained by PC1:", round(eigvals[0] / eigvals.sum(), 3))

In [ ]:
# ── 2.2 Plot the data and its first principal axis ──
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(small[:, 0], small[:, 1], color="tab:blue")
ax.scatter(*mean, color="green", marker="s", s=60, label="mean")
direction = eigvecs[:, 0]
t = np.linspace(-2, 2, 2)
line = mean[:, None] + direction[:, None] * t
ax.plot(line[0], line[1], "r-", label="PC1 (direction of max variance)")
ax.set_aspect("equal"); ax.legend(); ax.set_title("First principal component")
plt.show()

🔍 **What just happened?** PC1 runs along the direction the data is most
spread out in — not necessarily along either original axis. Projecting every point onto this one line
(rather than keeping both original coordinates) keeps as much of the data's variance as any single
direction possibly could, which is exactly PCA's optimality guarantee.

✏️ **Your turn.** Call `pca(small, n_components=1)` (the toolkit version) and
compare its returned `components` to `eigvecs[:, 0]` above — they should point along the same line (though
possibly flipped in sign, which does not matter: $w$ and $-w$ describe the same axis).

### 📘 Lecture notes — limitations of PCA

PCA is a good summary exactly when the interesting biological variation increases the variance of the data
along orthogonal directions. It can fail in at least three ways worth remembering: it may miss non-linear
patterns entirely (points lying along a curved manifold rather than a line/plane); structure that is not
aligned with (orthogonal to) the top components may be poorly captured; and PC1 maximizes variance, *not*
cluster separation — two obvious clusters can sit side-by-side along a direction PCA considers
unimportant.

## 3. Eigenimages and PCA for recognition

When PCA is applied directly to a stack of *images* (each image flattened into one long pixel vector), the
principal components themselves can be reshaped back into images — "eigenimages." A new image can then be
projected onto a handful of eigenimages and represented by just those few coefficients instead of every raw
pixel, which is both a compression and a feature-extraction step (this is the same idea behind "eigenfaces"
in face recognition, applied here to simulated nuclei).

In [ ]:
# ── 3.1 Build a bank of simulated nuclei images and compute their eigenimages ──
bank_images, bank_labels = [], []
for shape in SHAPE_CLASSES:
    for k in range(15):
        sim = simulate_nuclei_image(n_cells=1, size=32, shape_classes=(shape,), seed=10 + k)
        bank_images.append(sim["image"].ravel())
        bank_labels.append(shape)
X_bank = np.stack(bank_images)   # (N, 1024) -- each row is one flattened 32x32 image
print("image bank:", X_bank.shape)

scores, components, explained = pca(X_bank, n_components=4)
print("variance explained by the first 4 components:", np.round(explained, 3),
      " (cumulative:", round(explained.sum(), 3), ")")

In [ ]:
# ── 3.2 Visualize the eigenimages themselves ──
eigenimages = [components[i].reshape(32, 32) for i in range(4)]
show_image_grid(eigenimages, titles=[f"eigenimage {i+1} ({explained[i]:.0%} var.)" for i in range(4)],
                 ncols=4, cmap="RdBu_r")
plt.show()

🔍 **What just happened?** Each eigenimage is a *pattern of pixel
co-variation*, not a picture of any single cell — the first eigenimage is usually something like
"bright center vs. dim surround" (overall size/intensity), and later ones pick out finer shape contrasts.
Any of the 90 images in `X_bank` can now be approximately reconstructed as the mean image plus a weighted
sum of just these 4 eigenimages (`scores[i] @ components`), instead of needing all 1024 raw pixel
values.

In [ ]:
# ── 3.3 Do the top 2 PCA scores already separate shape classes? ──
fig, ax = plt.subplots(figsize=(5, 4))
for shape in SHAPE_CLASSES:
    sel = [i for i, s in enumerate(bank_labels) if s == shape]
    ax.scatter(scores[sel, 0], scores[sel, 1], label=shape, s=18)
ax.set_xlabel("PC1 score"); ax.set_ylabel("PC2 score"); ax.legend(fontsize=7)
ax.set_title("PCA of raw pixels, colored by TRUE shape label")
plt.show()

🧪 **Ground-truth check.** Because every point's true `shape` label is known,
we can color the PCA plot by ground truth and directly see how much of the biologically meaningful
variation the first two components captured, versus how much classes overlap — information a real,
unlabeled screen would not have this easy access to.

## 4. Classical multidimensional scaling (MDS): building a shape space

Rather than starting from feature *vectors*, MDS starts from a matrix of pairwise **dissimilarities** $D$
(distances) between $N$ objects and asks: what low-dimensional coordinates would reproduce these distances
as closely as possible? Formally, MDS finds $Z \in \mathbb{R}^{N\times k}$ minimizing the "stress"

$$\text{stress}(Z) = \sum_{i<j} \big(D_{ij} - \lVert z_i - z_j\rVert\big)^2$$

Classical (Euclidean) MDS solves this in closed form via the eigendecomposition of a "double-centered"
Gram matrix derived from $D^2$; general MDS minimizes stress by gradient descent, which also allows
non-Euclidean dissimilarities. This is exactly the tool behind CellOrganizer-style "shape space"
constructions for cell and nuclear shape, where $D_{ij}$ is a shape-to-shape distance rather than an
ordinary Euclidean one.

In [ ]:
# ── 4.1 Compute pairwise shape-feature distances for a small gallery of nuclei ──
from sklearn.manifold import MDS, Isomap, TSNE

gallery_imgs, gallery_feats, gallery_labels = [], [], []
for shape in SHAPE_CLASSES:
    for k in range(8):
        sim = simulate_nuclei_image(n_cells=1, size=64, shape_classes=(shape,), seed=20 + k)
        mask = sim["instance_mask"] > 0
        sf = shape_features(mask)
        gallery_feats.append([sf["compactness"], sf["aspect_ratio"], sf["area"] / 1000.0])
        gallery_imgs.append(sim["image"]); gallery_labels.append(shape)
gallery_feats = np.array(gallery_feats)

# pairwise Euclidean distance in (compactness, aspect ratio, scaled area) space -- our "shape distance"
diff = gallery_feats[:, None, :] - gallery_feats[None, :, :]
D = np.sqrt((diff ** 2).sum(axis=2))
print("distance matrix shape:", D.shape)

In [ ]:
# ── 4.2 Classical MDS: recover 2-D coordinates from the distance matrix alone ──
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=SEED, normalized_stress=False)
coords_mds = mds.fit_transform(D)
fig, ax = plt.subplots(figsize=(5, 4))
for shape in SHAPE_CLASSES:
    sel = [i for i, s in enumerate(gallery_labels) if s == shape]
    ax.scatter(coords_mds[sel, 0], coords_mds[sel, 1], label=shape, s=24)
ax.legend(fontsize=7); ax.set_title("MDS shape space (from a distance matrix only)")
plt.show()

🔍 **What just happened?** MDS was given *only* the N×N distance matrix `D` —
never the original (compactness, aspect ratio, area) coordinates — and still reconstructed a 2-D map whose
point-to-point distances approximate the originals. This is exactly analogous to reconstructing a map of
cities from a table of pairwise driving distances alone, with no latitude/longitude given.

✏️ **Your turn.** MDS and PCA are closely related for Euclidean distances: run
`pca(gallery_feats, n_components=2)` and compare its scatter plot (colored by shape) to the MDS plot above
— are the two maps approximately the same shape, up to rotation/reflection? (They should be, since
classical MDS on Euclidean distances is mathematically equivalent to PCA on the original features.)

## 5. Isomap: following the manifold

MDS (and PCA) use **straight-line** Euclidean distance. If the data actually lies on a curved,
lower-dimensional surface embedded in a higher-dimensional space — a *manifold* — straight-line distance
can badly misjudge how "close" two points really are: a route that must bend around the manifold is
shorter than it looks in a straight line. **Isomap** fixes this by estimating **geodesic** distance (the
shortest path *along* the manifold): build a k-nearest-neighbour graph of the data, treat edge weights as
local Euclidean distances, run a shortest-path algorithm (e.g. Dijkstra's) to approximate geodesic
distances between every pair of points, and then apply classical MDS to that geodesic distance matrix.

In [ ]:
# ── 5.1 A manifold where straight-line distance is misleading: an S-curve ──
def make_s_curve(n_points, noise=0.03, seed=SEED):
    g = np.random.default_rng(seed)
    t = g.uniform(-1.5 * np.pi, 1.5 * np.pi, n_points)
    height = g.uniform(-1.0, 1.0, n_points)
    x = np.sin(t)
    y = height
    z = np.sign(t) * (np.cos(t) - 1)
    pts3d = np.stack([x, y, z], axis=1) + g.normal(0, noise, size=(n_points, 3))
    return pts3d, t   # t is the TRUE 1-D position along the curve -- our ground truth

pts3d, t_true = make_s_curve(400)
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(pts3d[:, 0], pts3d[:, 1], pts3d[:, 2], c=t_true, cmap="viridis", s=8)
ax.set_title("A simulated S-curve manifold\n(color = true position along the curve)")
plt.show()

In [ ]:
# ── 5.2 PCA vs. Isomap on the S-curve ──
scores_pca, _, _ = pca(pts3d, n_components=2)
iso = Isomap(n_neighbors=10, n_components=2)
coords_iso = iso.fit_transform(pts3d)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(scores_pca[:, 0], scores_pca[:, 1], c=t_true, cmap="viridis", s=8)
axes[0].set_title("PCA (straight-line distance)")
axes[1].scatter(coords_iso[:, 0], coords_iso[:, 1], c=t_true, cmap="viridis", s=8)
axes[1].set_title("Isomap (geodesic distance)")
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** We know the true 1-D position `t_true` along the
S-curve for every point (the simulator's parametrization). PCA, using straight-line distance, folds the two
"arms" of the S on top of each other because they are close in ordinary 3-D space; Isomap's geodesic
distance correctly recognizes that reaching one arm from the other requires traveling *along* the curve,
and so unrolls the S into a smooth 1-D gradient of color — exactly matching `t_true`.

### 📘 Lecture notes — Isomap in three steps

1. **Build a neighborhood graph**: connect each point to its $k$ nearest neighbors (or all neighbors
   within radius $\varepsilon$), with edge weights equal to local Euclidean distance.
2. **Estimate geodesic distances**: for every pair of points, find the shortest path through this graph
   (summing edge weights) — a reasonable approximation to the manifold's true geodesic distance *when the
   data is densely and evenly sampled from a smooth manifold*.
3. **Embed**: apply classical MDS to the resulting geodesic distance matrix.

Isomap's main limitations follow directly from step 1: it needs dense, roughly noiseless sampling of the
manifold, assumes the manifold has no holes and is (loosely) convex, and like MDS it is computationally
expensive at scale since it needs all pairwise shortest paths.

## 6. t-SNE: preserving local neighborhoods

**t-distributed stochastic neighbor embedding (t-SNE)** takes a different, probabilistic approach: in the
*original* high-dimensional space, it defines a probability $p_{ij}$ that point $i$ would pick point $j$ as
a neighbor (higher for nearby points, controlled by a bandwidth parameter called **perplexity**, which
behaves similarly to Isomap's neighborhood size $k$); in the *low*-dimensional embedding, it defines an
analogous probability $q_{ij}$ using a heavier-tailed Student-t distribution; and it moves the
low-dimensional points by gradient descent to minimize the KL divergence between the $P$ and $Q$
distributions. Where PCA explicitly tries to preserve *global* variance, t-SNE explicitly trades that away
to preserve *local* neighborhoods extremely well — which is why it is popular for visualizing cluster
structure, but its inter-cluster distances and overall layout should not be over-interpreted.

In [ ]:
# ── 6.1 t-SNE on the same shape-feature gallery as Section 4 ──
tsne = TSNE(n_components=2, perplexity=10, random_state=SEED, init="pca")
coords_tsne = tsne.fit_transform(gallery_feats)
fig, ax = plt.subplots(figsize=(5, 4))
for shape in SHAPE_CLASSES:
    sel = [i for i, s in enumerate(gallery_labels) if s == shape]
    ax.scatter(coords_tsne[sel, 0], coords_tsne[sel, 1], label=shape, s=24)
ax.legend(fontsize=7); ax.set_title(f"t-SNE (perplexity={tsne.perplexity})")
plt.show()

✏️ **Your turn.** Re-run with `perplexity=3` and then `perplexity=25`
(perplexity cannot exceed roughly $n_{\text{points}}-1$ per class here, so keep it well under 48). Notice
how the apparent tightness and separation of clusters changes with perplexity alone, even though the
underlying data never changed — this is the single most important caveat when reading any t-SNE plot.

### 📘 Lecture notes — PCA vs. t-SNE, side by side

PCA seeks a *global*, linear structure: the directions of greatest overall variance, which can place points
that are actually far apart on the data manifold close together in the projection if that maximizes
captured variance. t-SNE seeks to preserve *local* structure: points that were neighbors in high dimensions
stay neighbors in the embedding, at the cost of distorting global distances and cluster sizes — two
well-separated t-SNE clusters are not necessarily "more different" than two nearby ones; only the
local neighborhood relationships are faithfully preserved.

## 7. Autoencoders: non-linear dimensionality reduction

An **autoencoder** is a neural network trained, without any labels, to reconstruct its own input through a
narrow "bottleneck": an **encoder** compresses the input $x$ into a low-dimensional latent vector $z$, and
a **decoder** reconstructs $\hat{x}$ from $z$, with the whole network trained end-to-end to minimize a
reconstruction loss (e.g. $\lVert x-\hat{x}\rVert^2$). Because the latent dimension is far smaller than
the input, the only way to reconstruct well is for the encoder to discover a compact, non-redundant summary
of what matters in the data — exactly PCA's goal, but with the non-linear flexibility of a neural network.
Unlike PCA (which minimizes a *distance-preservation* objective), an autoencoder directly minimizes
*reconstruction error*, which typically lets it fit non-linear structure (e.g. our S-curve manifold) that
PCA cannot, usually at the cost of a latent space that is harder to interpret axis-by-axis.

In [ ]:
# ── 7.1 A tiny convolutional autoencoder for 32x32 simulated nuclei ──
X_ae = torch.from_numpy(X_bank.reshape(-1, 1, 32, 32).astype(np.float32))
labels_ae = bank_labels

class TinyAutoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),      # 32 -> 16
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),     # 16 -> 8
            nn.Flatten(), nn.Linear(16 * 8 * 8, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16 * 8 * 8), nn.ReLU(), nn.Unflatten(1, (16, 8, 8)),
            nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1), nn.ReLU(),   # 8 -> 16
            nn.ConvTranspose2d(8, 1, 4, stride=2, padding=1), nn.Sigmoid(), # 16 -> 32
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

ae = TinyAutoencoder(latent_dim=2)
opt = torch.optim.Adam(ae.parameters(), lr=2e-3)
X_norm = (X_ae - X_ae.min()) / (X_ae.max() - X_ae.min() + 1e-9)   # scale to [0, 1] for the sigmoid decoder

In [ ]:
# ── 7.2 Train the autoencoder (unsupervised: no shape labels used) ──
for epoch in range(60):
    perm = torch.randperm(len(X_norm))
    epoch_loss = 0.0
    for i in range(0, len(X_norm), 16):
        idx = perm[i:i + 16]
        opt.zero_grad()
        recon, _ = ae(X_norm[idx])
        loss = ((recon - X_norm[idx]) ** 2).mean()
        loss.backward(); opt.step()
        epoch_loss += loss.item() * len(idx)
    if epoch % 15 == 0 or epoch == 59:
        print(f"epoch {epoch+1:3d}: reconstruction MSE = {epoch_loss/len(X_norm):.4f}")

In [ ]:
# ── 7.3 Compare an original image to its autoencoder reconstruction ──
with torch.no_grad():
    recon_all, latent = ae(X_norm)
idx_show = [0, 15, 30, 45, 60, 75]   # one example from several different shape classes
originals = [X_norm[i, 0].numpy() for i in idx_show]
recons = [recon_all[i, 0].numpy() for i in idx_show]
show_image_grid(originals + recons,
                 titles=[labels_ae[i] for i in idx_show] + ["reconstruction"] * len(idx_show), ncols=6)
plt.show()

🔍 **What just happened?** The autoencoder never saw a single shape label —
it only ever tried to compress each image down to 2 numbers and reconstruct the original from just those 2
numbers. The fact that the reconstructions are recognizable at all means those 2 latent numbers capture
most of what distinguishes one simulated nucleus image from another.

In [ ]:
# ── 7.4 Is the 2-D latent space organized by shape, even though it was never told the shapes? ──
latent_np = latent.numpy()
fig, ax = plt.subplots(figsize=(5, 4))
for shape in SHAPE_CLASSES:
    sel = [i for i, s in enumerate(labels_ae) if s == shape]
    ax.scatter(latent_np[sel, 0], latent_np[sel, 1], label=shape, s=18)
ax.legend(fontsize=7); ax.set_xlabel("latent dim 1"); ax.set_ylabel("latent dim 2")
ax.set_title("Autoencoder's 2-D latent space, colored by TRUE shape (unsupervised)")
plt.show()

🧪 **Ground-truth check.** As in Sections 3 and 6, we only *color* by the true
shape label after the fact — the autoencoder itself was trained with no access to it. Clustering by color
in this plot is therefore genuine evidence that reconstruction alone is enough to recover biologically
meaningful structure, without ever being told what that structure is — exactly the promise (and appeal) of
unsupervised feature learning in image-based profiling pipelines like CellProfiler's feature-clustering
workflows, which group and reduce large, redundant feature sets down to a compact, information-preserving
summary before any downstream classification or clustering.

✏️ **Your turn.** Increase `latent_dim` to 4 in Section 7.1, retrain, and
check whether the reconstruction MSE (printed during training) drops. Then apply the PCA or t-SNE code from
earlier sections to the 4-D latent vectors to get back down to a 2-D plot for visualization — dimensionality
reduction methods are often *chained*: a higher-capacity non-linear encoder for compression, followed by a
linear or local method purely for the final 2-D visualization.

📌 **Key takeaways**

- Dimensionality reduction fights the curse of dimensionality, enables visualization, and can improve downstream classifier performance; feature *selection* keeps a subset of original features, feature *extraction* (PCA, Isomap, t-SNE, autoencoders) builds new ones.
- PCA finds the directions of maximum variance via eigendecomposition of the covariance matrix (equivalently, via SVD); eigenimages are simply principal components reshaped back into image form.
- Classical MDS reconstructs coordinates from a distance matrix alone and, for Euclidean distances, is mathematically equivalent to PCA.
- Isomap replaces straight-line distance with geodesic (along-the-manifold) distance via a k-NN graph and shortest paths, correctly unrolling curved manifolds that fool PCA/MDS.
- t-SNE optimizes for faithful *local* neighborhoods (via a perplexity parameter), at the cost of not preserving global distances — read cluster sizes and inter-cluster gaps with caution.
- Autoencoders generalize PCA non-linearly by directly minimizing reconstruction error with a neural network, discovering a compact, often informative latent space with no labels at all.

✅ **Checkpoint**

1. Why is PCA on Euclidean distances mathematically equivalent to classical MDS on the same distances?
   <details><summary>Answer</summary>Both ultimately reduce to the eigendecomposition of a (co)variance
   or Gram matrix built from the same pairwise relationships — MDS just starts one step further along,
   directly from distances rather than from the raw feature vectors that produced them.</details>
2. Give one reason Isomap could fail on a sparsely-sampled, noisy manifold.
   <details><summary>Answer</summary>The k-NN graph can create "shortcut" edges between points that are
   close in raw Euclidean distance but far apart along the true manifold (if the manifold folds back near
   itself), corrupting the shortest-path geodesic-distance estimates.</details>
3. Why should you distrust the *size* of a cluster, or the *gap* between two clusters, in a t-SNE plot?
   <details><summary>Answer</summary>t-SNE's objective (matching local neighbor probabilities) does not
   constrain global distances or density at all — perplexity and random initialization can both change
   apparent cluster size/separation without the underlying data changing.</details>

🧠 **Questions & discussion**

- CellProfiler-style pipelines often compute hundreds of features per cell before any dimensionality reduction. Why might feature *selection* (dropping redundant features) be preferred over feature *extraction* (PCA) in a setting where biologists want to interpret individual features directly?
- We colored every unsupervised embedding in this notebook by a ground-truth label the method itself never saw. In a real screen with no such ground truth, how would you decide whether an apparent cluster in a t-SNE or autoencoder-latent plot is biologically real versus a visualization artifact?
- Autoencoders and PCA both compress data through a bottleneck, but only PCA gives orthogonal, variance-ranked components. What do you give up, and what might you gain, by using an autoencoder's latent space instead of PCA scores as classifier input?

🚀 **Bonus — going further**

- Compute the reconstruction error of PCA with 2 components on `X_bank` (project then reconstruct using the components already computed in Section 3) and compare it numerically to the 2-D autoencoder's reconstruction error from Section 7 — which is lower, and why would you expect that?
- Try Isomap with `n_neighbors=3` instead of 10 on the S-curve in Section 5 — does a too-small neighborhood break the geodesic-distance approximation?
- Add a bit of label noise to `gallery_labels` purely for visualization (do not retrain anything) and confirm that none of the five methods in this notebook use labels at any point — dimensionality reduction, as covered here, is entirely unsupervised.

📝 **Lab note**

```text
Module 09 — Dimensionality reduction and reconstruction
Date: ____________
Variance explained by PCA components 1-2 (image bank): ____ + ____ = ____
Did Isomap separate the S-curve's two arms?  yes / no
t-SNE perplexity tried: ____  Did cluster appearance change with perplexity?  yes / no
Autoencoder final reconstruction MSE: ____
One pattern I would want to confirm against real labels, and why: ____________
```

📚 **References**

- Pearson, K. (1901). On lines and planes of closest fit to systems of points in space. *Philosophical Magazine*, 2(11), 559–572.
- Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer.
- Torgerson, W. S. (1952). Multidimensional scaling: I. Theory and method. *Psychometrika*, 17(4), 401–419.
- Tenenbaum, J. B., de Silva, V. & Langford, J. C. (2000). A global geometric framework for nonlinear dimensionality reduction. *Science*, 290(5500), 2319–2323.
- van der Maaten, L. & Hinton, G. (2008). Visualizing data using t-SNE. *Journal of Machine Learning Research*, 9, 2579–2605.
- Hinton, G. E. & Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504–507.
- Johnston, C. A. et al. / Peralta, D. & Saeys, Y. (2020). Robust unsupervised dimensionality reduction based on feature clustering for single-cell imaging data. *Applied Soft Computing*, 93, 106421.

➡️ **Next.** With compact, meaningful representations in hand, Module 10 turns to the question of *which pixels belong to which object in the first place*: classical, geometrical segmentation methods — thresholding, watershed, active contours, and graph cuts.